# Домашнее задание 10. NLP Задание 2

In [1]:
import os
import re
import warnings
import random
from collections import defaultdict
from typing import Dict, List, Tuple

import torch
import torch.nn.functional as F
import numpy as np
import numpy as np
import torch
from tqdm.notebook import tqdm
from transformers import GPT2LMHeadModel, GPT2Tokenizer

warnings.filterwarnings("ignore")

In [2]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)

### 1. Реализация стратегий выбора следующего слова

In [3]:
class Model:
    def __init__(self, model_name: str = "gpt2"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = GPT2LMHeadModel.from_pretrained(model_name).to(self.device)
        self.tokenizer = GPT2Tokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.vocab_size = self.tokenizer.vocab_size
        print(f"Model loaded on {self.device}")

    def greedy_sampling(self, logits: torch.Tensor) -> int:
        """ Выбирает токен с максимальной вероятностью """
        # torch.argmax возвращает индекс максимального значения
        return torch.argmax(logits).item()

    def random_sampling(self, logits: torch.Tensor) -> int:
        """ Выбирает токен случайным образом на основе распределения """
        # Применяем softmax, чтобы получить вероятности
        probs = F.softmax(logits, dim=-1)
        # torch.multinomial сэмплирует один индекс на основе весов
        return torch.multinomial(probs, num_samples=1).item()

    def _beam_search_generate(
        self,
        prompt: str,
        max_length: int,
        num_beams: int,
        no_repeat_ngram_size: int = 3
    ) -> str:
        """
        Генерирует текст с использованием Beam Search и защитой от повторений (N-gram blocking).
        """
        input_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(self.device)

        beams = [(0.0, input_ids)]
        finished_beams = []

        for _ in range(max_length):
            all_candidates = []

            for score, seq in beams:
                if seq[0, -1].item() == self.tokenizer.eos_token_id:
                    finished_beams.append((score, seq))
                    continue

                with torch.no_grad():
                    outputs = self.model(seq)

                logits = outputs.logits[0, -1, :]
                log_probs = F.log_softmax(logits, dim=-1)

                # N-gram Blocking (защита от зацикливания)
                # Если мы уже сгенерировали достаточно токенов, проверяем повторы
                if no_repeat_ngram_size > 0 and seq.shape[1] >= no_repeat_ngram_size:
                    last_n_gram_prefix = seq[0, -(no_repeat_ngram_size - 1):].tolist()
                    current_seq_list = seq[0].tolist()

                    # Проходим по всей последовательности и ищем, где этот префикс встречался раньше
                    for i in range(len(current_seq_list) - no_repeat_ngram_size + 1):

                        ngram = current_seq_list[i : i + no_repeat_ngram_size]
                        if ngram[:no_repeat_ngram_size-1] == last_n_gram_prefix:
                            banned_token = ngram[-1]
                            log_probs[banned_token] = -float('inf')

                top_log_probs, top_indices = torch.topk(log_probs, num_beams)

                for i in range(num_beams):
                    token_id = top_indices[i].item()
                    token_score = top_log_probs[i].item()

                    # Если токен был заблокирован (-inf), не добавляем такой луч
                    if token_score == -float('inf'):
                        continue

                    new_seq = torch.cat([seq, torch.tensor([[token_id]], device=self.device)], dim=1)
                    new_score = score + token_score
                    all_candidates.append((new_score, new_seq))

            if not all_candidates:
                break

            ordered_candidates = sorted(all_candidates, key=lambda x: x[0], reverse=True)
            beams = ordered_candidates[:num_beams]

        finished_beams.extend(beams)

        if not finished_beams:
            return prompt

        finished_beams.sort(key=lambda x: x[0], reverse=True)
        best_score, best_seq = finished_beams[0]

        return self.tokenizer.decode(best_seq[0], skip_special_tokens=True).strip()

    def apply_temperature(self, logits: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
        """ Температура, делает распределение более "острым" (temp < 1.0) или "плоским" (temp > 1.0) """

        if temperature == 0.0:
            return logits

        return logits / temperature

    def _apply_top_p(self, logits: torch.Tensor, top_p: float = 1.0) -> torch.Tensor:
        """ Top-p sampling. Оставляет только токены, чья кумулятивная вероятность > top_p. """

        if top_p >= 1.0 or top_p <= 0.0:
            return logits # top_p=1.0 - это отсутствие фильтрации

        # Сортируем вероятности по убыванию
        probs = F.softmax(logits, dim=-1)
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)

        # Считаем кумулятивную сумму
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

        sorted_indices_to_remove = cumulative_probs > top_p

        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = False # Никогда не удаляем первый (самый вероятный) токен

        # Получаем оригинальные индексы токенов для удаления
        indices_to_remove = sorted_indices[sorted_indices_to_remove]

        # "Выключаем" эти токены, устанавливая их логиты в -infinity
        logits[indices_to_remove] = -float('inf')

        return logits

    def _apply_top_k(self, logits: torch.Tensor, top_k: int = 0) -> torch.Tensor:
        """ Top-k sampling. Оставляет только k самых вероятных токенов. """

        # Если top_k=0, это значит "без фильтрации"
        if top_k <= 0 or top_k >= self.vocab_size:
            return logits

        kth_value = torch.topk(logits, top_k)[0][-1]

        # Устанавливаем все логиты, которые *меньше* k-го, в -infinity
        logits[logits < kth_value] = -float('inf')
        return logits

    def generate(
        self,
        prompt: str,
        max_length: int = 50,
        strategy: str = "greedy",
        temperature: float = 1.0,
        top_k: int = 0,
        top_p: float = 1.0,
        num_beams: int = 3
    ) -> str:
        """
        Главная функция генерации, управляющая всеми стратегиями.

        :param prompt: Начальный текст.
        :param max_length: Максимальная длина генерации (включая prompt).
        :param strategy: 'greedy', 'random' (для top-k/top-p) или 'beam'.
        :param temperature: Температура сэмплирования.
        :param top_k: Параметр Top-k.
        :param top_p: Параметр Top-p.
        :param num_beams: Количество лучей для 'beam' стратегии.
        :return: Сгенерированный текст.
        """

        # Переключаемся на beam search, если указано
        if strategy == "beam":
            return self._beam_search_generate(prompt, max_length, num_beams)

        # Токенизируем вход
        input_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(self.device)

        # Рассчитываем, сколько токенов нужно сгенерировать
        num_tokens_to_generate = max_length - input_ids.shape[1]
        if num_tokens_to_generate <= 0:
            return prompt

        for _ in range(num_tokens_to_generate):
            # Получаем логиты для последнего токена
            with torch.no_grad():
                outputs = self.model(input_ids)

            next_token_logits = outputs.logits[0, -1, :]

            # Модификация логитов
            next_token_logits = self.apply_temperature(next_token_logits, temperature)
            next_token_logits = self._apply_top_k(next_token_logits, top_k)
            next_token_logits = self._apply_top_p(next_token_logits, top_p)

            # Сэмплирование
            if strategy == "greedy":
                next_token_id = self.greedy_sampling(next_token_logits)
            elif strategy == "random":
                next_token_id = self.random_sampling(next_token_logits)
            else:
                raise ValueError(f"Unknown generation strategy: {strategy}")

            # Добавляем сгенерированный токен к последовательности
            input_ids = torch.cat(
                [input_ids, torch.tensor([[next_token_id]], device=self.device)],
                dim=1
            )

            if next_token_id == self.tokenizer.eos_token_id:
                break

        return self.tokenizer.decode(input_ids[0], skip_special_tokens=True).strip()



### 2.  Тестирование

In [4]:
def test_generation_strategies(model, prompt: str, max_length=30):
    """
    Запускает тесты различных стратегий генерации для заданной модели и промта.
    """
    print("---" * 10)
    print(f"Prompt: {prompt}\n")

    # 1. Greedy
    print("Strategy: Greedy")
    generated_text = model.generate(prompt, max_length=max_length, strategy="greedy")
    print(f"Result: {generated_text}\n")

    # 2. Random Sampling (c температурой)
    print("Strategy: Random (temp=1.5)")
    generated_text = model.generate(
        prompt, max_length=max_length, strategy="random", temperature=1.5
    )
    print(f"Result: {generated_text}\n")

    # 3. Random Sampling (Top-k)
    print("Strategy: Random (top_k=50)")
    generated_text = model.generate(
        prompt, max_length=max_length, strategy="random", top_k=50
    )
    print(f"Result: {generated_text}\n")

    # 4. Random Sampling (Top-p)
    print("Strategy: Random (top_p=0.9)")
    generated_text = model.generate(
        prompt, max_length=max_length, strategy="random", top_p=0.9
    )
    print(f"Result: {generated_text}\n")

    # 5. Beam Search
    print("Strategy: Beam (num_beams=4)")
    generated_text = model.generate(
        prompt, max_length=max_length, strategy="beam", num_beams=4
    )
    print(f"Result: {generated_text}\n")

In [5]:
model = Model(model_name="gpt2") # "sberbank-ai/rugpt3small_based_on_gpt2" для русского

Model loaded on cuda


In [6]:
prompt = "Once upon a time,"

test_generation_strategies(model, prompt, max_length=25)

------------------------------
Prompt: Once upon a time,

Strategy: Greedy
Result: Once upon a time, the world was a place of great beauty and great danger. The world was a place of great danger

Strategy: Random (temp=1.5)
Result: Once upon a time, none the station that protagonist Akira Celvin wished had storage space in Speechorial Mjuush instead

Strategy: Random (top_k=50)
Result: Once upon a time, I could do nothing for her… I knew this for a few moments, then immediately felt myself trembling

Strategy: Random (top_p=0.9)
Result: Once upon a time, however, the President perceived Mr. Ford's bemusement about the press and his remarks —

Strategy: Beam (num_beams=4)
Result: Once upon a time, it was said, there would be a time when the world would cease to exist.

And now, it is said



In [7]:
prompt = "Please don't forget"

test_generation_strategies(model, prompt)

------------------------------
Prompt: Please don't forget

Strategy: Greedy
Result: Please don't forget to share this post with your friends and family.

This post is part of the "The Best of the Best" series

Strategy: Random (temp=1.5)
Result: Please don't forgetThings.dd. TyDrYC KimBi66 ScotlandSince athenium Depoupper RIPAboutDE OF FACE released Stellar's

Strategy: Random (top_k=50)
Result: Please don't forget to like us on Facebook.

Want to learn more about all things tech like Amazon, Google Play Store, Netflix, Google

Strategy: Random (top_p=0.9)
Result: Please don't forget to share this article in the future as the list is evolving and new information is collected.

Social Monitoring Platform Updates

Strategy: Beam (num_beams=4)
Result: Please don't forget to follow us on Twitter and like us on Facebook.



In [16]:
prompt = "And then the wizard cast a"

test_generation_strategies(model, prompt)

------------------------------
Prompt: And then the wizard cast a

Strategy: Greedy
Result: And then the wizard cast a spell that would make the wizard's body disappear.

The wizard's body was then magically transformed into a giant,

Strategy: Random (temp=1.5)
Result: And then the wizard cast a talisman at her designate (granting HIGH FREEST MONSTERS TIN AND CAGE), printed one record withstand

Strategy: Random (top_k=50)
Result: And then the wizard cast a magic circle, revealing that it was to use them to make a crossbow and as this, a powerful bolt, the

Strategy: Random (top_p=0.9)
Result: And then the wizard cast a spell. The spell was a little easier than the spell had been originally intended to be, but the wizard was able to

Strategy: Beam (num_beams=4)
Result: And then the wizard cast a spell.

The wizard cast the spell, and then the spell was cast. The wizard cast another spell, but this time it was the same



In [9]:
ru_model = Model(model_name="sberbank-ai/rugpt3small_based_on_gpt2") # "sberbank-ai/rugpt3small_based_on_gpt2" для русского

Model loaded on cuda


In [10]:
prompt = "Давным-давно"

test_generation_strategies(ru_model, prompt)

------------------------------
Prompt: Давным-давно

Strategy: Greedy
Result: Давным-давно, когда я был еще совсем маленьким, я был очень любопытен.  И вот однажды, когда я был совсем маленьким,

Strategy: Random (temp=1.5)
Result: Давным-давно. Билетов ленинградской авиакомпании «МИБ». Нельден Роэм Беренд подыскали Ленни Прилучная Виктора Ку

Strategy: Random (top_k=50)
Result: Давным-давно давным-давно, когда все так жили, на Земле жил самый древний человек по имени Амос (или по-гречески

Strategy: Random (top_p=0.9)
Result: Давным-давно они жили на самом берегу моря, вдалеке от многолюдной среды и шума города.  А сейчас, когда многие гордые

Strategy: Beam (num_beams=4)
Result: Давным-давно, давным-давно…


* * *

— Ну, что? — спросил я, когда мы вышли на улицу.



In [11]:
prompt = "Британские ученые доказали, что"

test_generation_strategies(ru_model, prompt)

------------------------------
Prompt: Британские ученые доказали, что

Strategy: Greedy
Result: Британские ученые доказали, что в организме человека есть вещества, которые могут вызывать рак.  В частности, они обнаружили, что в организме человека есть

Strategy: Random (temp=1.5)
Result: Британские ученые доказали, что зелень древнетенчих сроков пастбищ важна видом соска привлечет пчелносиц Ален Перимео...

Strategy: Random (top_k=50)
Result: Британские ученые доказали, что в атмосфере земли на поверхности Земли имеются газы. 
 В течение 20 лет исследователи исследовали на Большом адронном

Strategy: Random (top_p=0.9)
Result: Британские ученые доказали, что солнце, о котором я вам рассказывала, является оптическим обманом, и что атомные ракеты не являются отражением

Strategy: Beam (num_beams=4)
Result: Британские ученые доказали, что у людей, страдающих от диабета, может развиться диабет второго типа.  Об этом сообщает The Daily Mail. 
 Ученые обнаружили, что



In [12]:
prompt = "Искусственный интеллект захватит мир, когда"

test_generation_strategies(ru_model, prompt)

------------------------------
Prompt: Искусственный интеллект захватит мир, когда

Strategy: Greedy
Result: Искусственный интеллект захватит мир, когда он будет готов к войне.

Strategy: Random (temp=1.5)
Result: Искусственный интеллект захватит мир, когда несчастья, война усилится… маленькая трансляция осуществмана гораздо умела учитель писал армотереизмышления одними

Strategy: Random (top_k=50)
Result: Искусственный интеллект захватит мир, когда захочет.  Но тогда и нам не понадобится ничего, чтобы стать частью этого мира", - сообщил министр иностранных

Strategy: Random (top_p=0.9)
Result: Искусственный интеллект захватит мир, когда все расставит по своим местам, а сигнал будет соваться в его сердечную клетку изнутри его плоти.

Strategy: Beam (num_beams=4)
Result: Искусственный интеллект захватит мир, когда он будет уничтожен.

— Да, — согласился я. — Но это не значит, что мы не сможем его уничтожить.

